# 🔍 Anomaly Detection in Transaction Data

This notebook demonstrates anomaly detection using the **Isolation Forest** algorithm on transaction data.

## Pipeline
1. Load data from MySQL
2. Explore & visualize distributions
3. Train Isolation Forest model
4. Evaluate and visualize results
5. Store anomalies back to MySQL

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sqlalchemy import create_engine, text
import sys, os

# Add scripts dir to path for config
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
from config import get_connection_string

sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries loaded ✅')

## 1. Load Data from MySQL

In [ ]:
engine = create_engine(get_connection_string())
df = pd.read_sql('SELECT * FROM raw_transactions', engine)
print(f'Loaded {len(df)} transactions')
df.head(10)

## 2. Exploratory Data Analysis

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Amount distribution
axes[0].hist(df['amount'].dropna(), bins=50, color='#3b82f6', edgecolor='white', alpha=0.8)
axes[0].set_title('Transaction Amount Distribution', fontweight='bold')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Frequency')

# Amount boxplot
axes[1].boxplot(df['amount'].dropna(), vert=True)
axes[1].set_title('Amount Boxplot', fontweight='bold')
axes[1].set_ylabel('Amount ($)')

# Region counts
df['region'].value_counts().plot(kind='bar', ax=axes[2], color='#8b5cf6', edgecolor='white')
axes[2].set_title('Transactions by Region', fontweight='bold')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Train Isolation Forest Model

In [ ]:
# Prepare features — only use valid amounts
df_valid = df[df['amount'].notna() & (df['amount'] > 0)].copy()
X = df_valid[['amount']].values

print(f'Training set size: {len(X)}')
print(f'Amount range: ${X.min():.2f} – ${X.max():.2f}')
print(f'Mean amount: ${X.mean():.2f}')

In [ ]:
# Train model
model = IsolationForest(
    n_estimators=100,
    contamination=0.02,
    random_state=42,
)
model.fit(X)

# Predict
df_valid['prediction'] = model.predict(X)
df_valid['anomaly_score'] = model.decision_function(X)

n_anomalies = (df_valid['prediction'] == -1).sum()
n_normal = (df_valid['prediction'] == 1).sum()
print(f'\nResults:')
print(f'  Normal     : {n_normal}')
print(f'  Anomalies  : {n_anomalies}')
print(f'  Anomaly %  : {100 * n_anomalies / len(df_valid):.1f}%')

## 4. Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#ef4444' if p == -1 else '#10b981' for p in df_valid['prediction']]

# Scatter: amount vs anomaly score
axes[0].scatter(df_valid['amount'], df_valid['anomaly_score'], c=colors, alpha=0.6, s=30)
axes[0].set_xlabel('Amount ($)', fontsize=12)
axes[0].set_ylabel('Anomaly Score', fontsize=12)
axes[0].set_title('Anomaly Score vs Transaction Amount', fontweight='bold', fontsize=14)
axes[0].axhline(y=0, color='#f59e0b', linestyle='--', alpha=0.5, label='Decision boundary')
axes[0].legend()

# Anomaly amount distribution
anomalies = df_valid[df_valid['prediction'] == -1]
normals = df_valid[df_valid['prediction'] == 1]
axes[1].hist(normals['amount'], bins=40, color='#10b981', alpha=0.6, label=f'Normal ({len(normals)})', edgecolor='white')
axes[1].hist(anomalies['amount'], bins=20, color='#ef4444', alpha=0.8, label=f'Anomaly ({len(anomalies)})', edgecolor='white')
axes[1].set_xlabel('Amount ($)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Amount Distribution: Normal vs Anomaly', fontweight='bold', fontsize=14)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Show the top anomalies
print('Top 15 Anomalies (sorted by score):')
top_anomalies = anomalies.nsmallest(15, 'anomaly_score')[['transaction_id', 'amount', 'region', 'transaction_date', 'anomaly_score']]
top_anomalies

## 5. Anomaly Distribution by Region

In [ ]:
if len(anomalies) > 0:
    region_counts = anomalies['region'].value_counts()
    fig, ax = plt.subplots(figsize=(10, 5))
    region_counts.plot(kind='bar', color='#ef4444', edgecolor='white', alpha=0.8, ax=ax)
    ax.set_title('Anomalies by Region', fontweight='bold', fontsize=14)
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('No anomalies detected.')

## 6. Model Parameters & Experimentation

Try different contamination rates to see how the model reacts:

In [ ]:
contamination_rates = [0.01, 0.02, 0.05, 0.10]
results = []

for rate in contamination_rates:
    m = IsolationForest(n_estimators=100, contamination=rate, random_state=42)
    m.fit(X)
    preds = m.predict(X)
    n_anom = (preds == -1).sum()
    results.append({'contamination': rate, 'anomalies_found': n_anom, 'anomaly_pct': f'{100*n_anom/len(X):.1f}%'})

pd.DataFrame(results)

## 7. Save Results to MySQL

In [ ]:
# Store anomalies in the database
with engine.connect() as conn:
    conn.execute(text('DELETE FROM anomalies'))
    conn.commit()

anomaly_records = anomalies[['transaction_id', 'anomaly_score']].copy()
anomaly_records['detection_method'] = 'IsolationForest'
anomaly_records.to_sql('anomalies', engine, if_exists='append', index=False)

print(f'✅ Stored {len(anomaly_records)} anomalies in MySQL (anomalies table)')

---
### Summary

| Metric | Value |
|--------|-------|
| Training Samples | `len(X)` |
| Algorithm | Isolation Forest |
| Contamination | 2% |
| Anomalies Found | `n_anomalies` |

The anomalies are primarily **high-value transactions** that deviate significantly from the normal amount distribution.